In [8]:
import sys
sys.path.insert(0, r'C:\Users\USER\Projects\TradeFlow\src')
from forecasting import preview_forecasts

# See all forecasts
preview_forecasts()

# See just Yam forecasts
preview_forecasts(commodity_name="Yam")

# See just Lagos forecasts
preview_forecasts(state_name="Lagos")


  FORECAST PREVIEW — Generated 2026-04-13
forecast_date    state commodity predicted_price lower_bound upper_bound        risk
   2026-04-13    Abuja     Maize         ₦28,096     ₦26,804     ₦29,338    ✓ Normal
   2026-04-14    Abuja     Maize         ₦28,839     ₦27,645     ₦30,175    ✓ Normal
   2026-04-15    Abuja     Maize         ₦29,967     ₦28,808     ₦31,154    ✓ Normal
   2026-04-16    Abuja     Maize         ₦31,445     ₦30,011     ₦32,763 ⚠ HIGH RISK
   2026-04-17    Abuja     Maize         ₦33,780     ₦32,531     ₦35,107 ⚠ HIGH RISK
   2026-04-18    Abuja     Maize         ₦37,119     ₦35,920     ₦38,352 ⚠ HIGH RISK
   2026-04-19    Abuja     Maize         ₦38,764     ₦37,288     ₦40,017 ⚠ HIGH RISK
   2026-04-20    Abuja     Maize         ₦43,428     ₦42,185     ₦44,680 ⚠ HIGH RISK
   2026-04-13     Kogi     Maize         ₦29,978     ₦28,384     ₦31,552    ✓ Normal
   2026-04-14     Kogi     Maize         ₦28,971     ₦27,508     ₦30,660    ✓ Normal
   2026-04-15     Kogi

,forecast_date,state,commodity,predicted_price,lower_bound,upper_bound,risk
0,2026-04-13,Lagos,Maize,"₦37,619","₦35,795","₦39,540",✓ Normal
1,2026-04-14,Lagos,Maize,"₦33,956","₦32,260","₦35,712",✓ Normal
2,2026-04-15,Lagos,Maize,"₦33,612","₦31,855","₦35,338",✓ Normal
3,2026-04-16,Lagos,Maize,"₦27,969","₦26,190","₦29,814",⚠ HIGH RISK
4,2026-04-17,Lagos,Maize,"₦22,656","₦20,947","₦24,423",⚠ HIGH RISK
5,2026-04-18,Lagos,Maize,"₦15,206","₦13,289","₦16,838",⚠ HIGH RISK
6,2026-04-19,Lagos,Maize,"₦5,933","₦3,878","₦7,767",⚠ HIGH RISK
7,2026-04-20,Lagos,Maize,₦0,₦0,₦0,⚠ HIGH RISK
8,2026-04-13,Lagos,Rice,"₦49,332","₦46,932","₦51,774",✓ Normal
9,2026-04-14,Lagos,Rice,"₦47,573","₦45,259","₦50,092",⚠ HIGH RISK


In [5]:
import sqlite3, pandas as pd
conn = sqlite3.connect(r'C:\Users\USER\Projects\TradeFlow\data\tradeflow.db')
df = pd.read_sql("""
    SELECT MIN(price_date) as earliest,
           MAX(price_date) as latest,
           COUNT(*) as total
    FROM cleaned_prices
""", conn)
print(df)
conn.close()

     earliest      latest  total
0  2026-02-15  2026-04-05    306


In [7]:
import sqlite3, numpy as np, pandas as pd
from datetime import date, timedelta

conn = sqlite3.connect(r'C:\Users\USER\Projects\TradeFlow\data\tradeflow.db')
conn.execute("DELETE FROM cleaned_prices")
conn.commit()

np.random.seed(42)
base_prices = {
    1: {6:18000, 7:16000, 8:17000, 4:20000, 5:21000, 2:25000, 1:28000, 3:26000},
    2: {6:28000, 7:26000, 8:27000, 4:30000, 5:31000, 2:35000, 1:38000, 3:36000},
    3: {6:45000, 7:43000, 8:44000, 4:47000, 5:48000, 2:52000, 1:55000, 3:53000},
    4: {6: 3500, 7: 3200, 8: 3300, 4: 3800, 5: 4000, 2: 5500, 1: 6500, 3: 6000},
}

records = []

# Generate one entry per day for the past 56 days ending TODAY
for days_ago in range(55, -1, -1):
    price_date = date.today() - timedelta(days=days_ago)
    for commodity_id, state_prices in base_prices.items():
        for state_id, base in state_prices.items():
            noise = np.random.normal(0, base * 0.05)
            price = round(max(base + noise, base * 0.7), 2)
            records.append((
                state_id, commodity_id,
                price, price / 100,
                price_date, False, True
            ))

conn.executemany("""
    INSERT OR IGNORE INTO cleaned_prices
    (state_id, commodity_id, price_per_unit, price_per_kg,
     price_date, is_outlier, is_confirmed)
    VALUES (?, ?, ?, ?, ?, ?, ?)
""", records)
conn.commit()

df = pd.read_sql("""
    SELECT MIN(price_date) as earliest,
           MAX(price_date) as latest,
           COUNT(*) as total
    FROM cleaned_prices
""", conn)
print(df)
conn.close()

     earliest      latest  total
0  2026-02-17  2026-04-13   1792
